# Concept-Aware Training — Research Tasks 5–8 (REVISED)

**July 2026 rerun with validity fixes.** The June round (`research_tasks_5_8.ipynb`) surfaced promising numbers, but a code/data audit found problems that invalidate several of them:

| # | Problem | Impact | Fix in this notebook |
|---|---------|--------|----------------------|
| 1 | **Train/val leakage**: 220/222 syn-val contexts appear in hyp-train — the merged contrastive model *trained on ~99% of its syn eval contexts* | Merged concept PPL **1.37 is memorization**, not generalization; merged-vs-syn-only gap explained by the leak | `rebuild_clean_splits.py`: splits grouped by source context; **all models retrained** on clean splits |
| 2 | **Wrong-sense mining degenerate**: old code defined "intended senses" as *every* synset of every positive → wrong-sense set empty by construction (0% coverage) | "Wrong-sense only" ablation was actually a **no-negatives run** | Synonym-intersection sense disambiguation in `build_contrastive_dataset.py`; real coverage now ~86–88%; explicit `none` control kept |
| 3 | **Circular eval metric**: v1 scored bare-word token ids (no leading space) — the exact convention the concept trainers optimize; CLM never trained on it | Concept-PPL gap over CLM partially measured "did the model learn our token convention" | `eval_concept_ppl_v2.py`: in-context continuation scoring |
| 4 | **Eval-time embedding resize** + per-checkpoint tokenizers | Checkpoints compared under non-identical softmax distributions (tied embeddings!) | v2: one canonical tokenizer, never resizes |
| 5 | **Single-token filter** (72–78% slot coverage) + no uncertainty | Multi-token concepts silently dropped; no error bars | v2: multi-token scoring, 95% bootstrap CIs, coverage stats disambiguated |

**Numbers from this notebook are NOT comparable to the June round** — different splits, different metric convention. That is the point.

### Budget-ordered plan (Colab Pro, limited)

| Priority | Phase | What | Approx cost |
|---|---|---|---|
| 1 | 1 | Split rebuild + dataset rebuild + eval rewrite | ~0 (CPU) |
| 2 | 2–5 | Retrain all models on clean splits, revised eval, ablations | few GPU-hours |
| 3 | 6 | One insurance seed of the headline model | ~1 GPU-hour |
| 4 | 7 | Downstream redesign: linear probe primary + low-resource FT | moderate |
| 5 | — | Full 3-seed × 3-model grid | deferred to paper-writing |
| 6 | — | Pythia-1.4B | last, per Chen |

**Before running:** `Runtime > Change runtime type > GPU` (T4 or better). Make sure the repo on GitHub has the fixed scripts (`rebuild_clean_splits.py`, updated `build_contrastive_dataset.py`, `eval_concept_ppl_v2.py`).


---
## 0. Setup

In [ ]:
import subprocess, torch
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout)
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
!pip install -q "transformers>=4.41.0,<5.0.0" datasets accelerate evaluate nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/SharvaGogawale1/concept-aware-training.git'
REPO_DIR = '/content/concept_aware_training'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

%cd {REPO_DIR}

# The revised round depends on the July 2026 fixes — verify they are present.
for required in ['rebuild_clean_splits.py',
                 'transformers/examples/pytorch/language-modeling/eval_concept_ppl_v2.py']:
    assert os.path.exists(os.path.join(REPO_DIR, required)), \
        f'MISSING: {required} — push the fixed scripts to the repo first.'
print('Repo ready (revised scripts present).')

In [ ]:
import os, json
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_DIR    = '/content/concept_aware_training'
DATA_ROOT   = f'{REPO_DIR}/data'
SCRIPTS_DIR = f'{REPO_DIR}/transformers/examples/pytorch/language-modeling'
OUTPUT_ROOT = '/content/drive/MyDrive/concept_aware_outputs'   # <-- UPDATE if needed
CLEAN_OUT   = f'{OUTPUT_ROOT}/clean'          # retrained checkpoints live here
RESULTS_DIR = f'{OUTPUT_ROOT}/clean_results'  # all result JSONs live here
MODEL_LOCAL_PATH = '/content/Llama-3.2-1B'
TOKENIZER_PATH   = MODEL_LOCAL_PATH           # canonical tokenizer for ALL evals

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CLEAN_OUT, exist_ok=True)

# ── Clean data (produced by Phase 1) ─────────────────────────────────────────
CLEAN_SYN = f'{DATA_ROOT}/syn/youtube_clean'
CLEAN_HYP = f'{DATA_ROOT}/hyp/youtube_clean'

SYN_CONCEPT_TRAIN_C = f'{CLEAN_SYN}/context_loss_train.csv'
SYN_CONCEPT_VAL_C   = f'{CLEAN_SYN}/context_loss_val.csv'
HYP_CONCEPT_TRAIN_C = f'{CLEAN_HYP}/context_loss_train.csv'
HYP_CONCEPT_VAL_C   = f'{CLEAN_HYP}/context_loss_val.csv'
HYP_DICT_TRAIN_C    = f'{CLEAN_HYP}/dict_loss_train.csv'
HYP_DICT_VAL_C      = f'{CLEAN_HYP}/dict_loss_val.csv'
SYN_TXT_TRAIN_C     = f'{CLEAN_SYN}/context_syn_train.txt'
SYN_TXT_VAL_C       = f'{CLEAN_SYN}/context_syn_val.txt'
VANILLA_VAL_C       = f'{CLEAN_HYP}/vanilla_val.txt'

# ── Contrastive datasets (built in Phase 1 from the clean splits) ────────────
CONTRASTIVE_ROOT = f'{DATA_ROOT}/contrastive'
CONTRA_DIRS = {
    'merged':   f'{CONTRASTIVE_ROOT}/youtube_clean',
    'syn_only': f'{CONTRASTIVE_ROOT}/youtube_clean_syn_only',
    'hyp_only': f'{CONTRASTIVE_ROOT}/youtube_clean_hyp_only',
}
ABLATION_STRATEGIES = ['co_hyponym', 'wrong_sense', 'same_pos', 'none']
for s in ABLATION_STRATEGIES:
    CONTRA_DIRS[f'ablation_{s}'] = f'{CONTRASTIVE_ROOT}/youtube_clean_ablation_{s}'

# ── Checkpoints (Phase 2+; every model retrained on the clean splits) ────────
CKPTS = {
    'clm':                  f'{CLEAN_OUT}/standard_clm',
    'syn_ncp':              f'{CLEAN_OUT}/syn_ncp',
    'hyp_ncp':              f'{CLEAN_OUT}/hyp_ncp',
    'diff_ncp':             f'{CLEAN_OUT}/diff_ncp',
    'contrastive_merged':   f'{CLEAN_OUT}/contrastive_merged',
    'contrastive_syn_only': f'{CLEAN_OUT}/contrastive_syn_only',
    'contrastive_hyp_only': f'{CLEAN_OUT}/contrastive_hyp_only',
}
for s in ABLATION_STRATEGIES:
    CKPTS[f'ablation_{s}'] = f'{CLEAN_OUT}/contrastive_ablation_{s}'
CKPTS['contrastive_merged_seed123'] = f'{CLEAN_OUT}/contrastive_merged_seed123'

LABELS = {
    'clm':                  'Standard CLM',
    'syn_ncp':              'Synonym NCP (original objective)',
    'hyp_ncp':              'Hypernym NCP (original objective)',
    'diff_ncp':             'Differentiable NCP (α=1.0)',
    'contrastive_merged':   'Contrastive — merged syn+hyp',
    'contrastive_syn_only': 'Contrastive — syn-only',
    'contrastive_hyp_only': 'Contrastive — hyp-only',
    'ablation_co_hyponym':  'Contrastive — co-hyponym only',
    'ablation_wrong_sense': 'Contrastive — wrong-sense only (FIXED)',
    'ablation_same_pos':    'Contrastive — same-POS only',
    'ablation_none':        'Contrastive — NO negatives (control)',
    'contrastive_merged_seed123': 'Contrastive — merged (seed 123)',
}

def existing(keys):
    return {k: CKPTS[k] for k in keys if os.path.exists(CKPTS[k])}

print('Paths configured. Existing checkpoints:')
for k, v in CKPTS.items():
    print(f"  {'✅' if os.path.exists(v) else '—':2s} {k}: {v}")

In [ ]:
# Resume-after-disconnect diagnostic
# Drive-backed paths (CKPTS, RESULTS_DIR) survive a runtime disconnect.
# Local Colab disk (REPO_DIR, MODEL_LOCAL_PATH, nltk data, data/*_clean) does not
# and must be regenerated by re-running Setup + Phase 1 before this cell is useful.
# Run this any time you reconnect to see exactly what to skip vs. re-run.

def _status(path, label):
    ok = os.path.exists(path)
    if ok and os.path.isdir(path):
        ok = len(os.listdir(path)) > 0
    print(f"{'✅' if ok else '—':2s} {label}")
    return ok

print('=== Phase 2: checkpoints ===')
for k, v in CKPTS.items():
    _status(v, k)

print('\n=== Phase 3/6/8: dual-eval result JSONs ===')
for name in ['dual_eval_v2_syn.json', 'dual_eval_v2_hyp.json',
             'task8R_ablation_syn.json', 'task8R_ablation_hyp.json',
             'seed_check.json']:
    _status(f'{RESULTS_DIR}/{name}', name)

print('\n=== Phase 7: downstream result JSONs ===')
for name in ['snli_probe_v2.json', 'snli_lowres_100.json',
             'snli_lowres_500.json', 'snli_lowres_1000.json',
             'spam_probe_v2.json']:
    _status(f'{RESULTS_DIR}/{name}', name)


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="meta-llama/Llama-3.2-1B",
    local_dir="/content/Llama-3.2-1B",
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
)
print("Base model ready at /content/Llama-3.2-1B")

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('WordNet ready.')

---
## Phase 1 — Data fixes (Priority 1, ~0 cost, CPU only)

### 1a. Rebuild leak-free splits

The June splits leak: **220/222 unique syn-val contexts appear in hyp-train** (`syn_val ∩ syn_train` was 0, so the syn-only comparison was clean, but every model that touched hyp data saw the syn val set). `rebuild_clean_splits.py` pools all contexts, groups prefix-nested contexts of the same source sentence (plus `<mask>`-style rows attached by tail matching), splits at the *group* level, and **hard-fails if any residual overlap remains**.

In [ ]:
%cd {REPO_DIR}
!python rebuild_clean_splits.py --data_root data --in_subdir youtube --out_subdir youtube_clean --val_frac 0.12 --seed 42

# The script exits non-zero on residual leakage; make failure loud in Colab too.
report = json.load(open(f'{DATA_ROOT}/youtube_clean_split_report.json'))
leaks = report['after']['leak_audit']
assert all(v == 0 for v in leaks.values()), f'LEAKAGE REMAINS: {leaks}'
print('\nLeak audit:', leaks)
print('Split sizes:', {k: v for k, v in report['after'].items() if k != 'leak_audit'})

### 1b. Rebuild contrastive datasets from the clean splits

Uses the **fixed** `build_contrastive_dataset.py`:
- wrong-sense mining now disambiguates the intended synset via synonym intersection (was 0% coverage — structurally empty; now ~86–88%);
- hierarchy-related senses (hypernym/hyponym of the intended sense) are excluded from wrong-sense negatives;
- `none` strategy = explicit no-negatives control (the June "wrong-sense" run was accidentally this);
- prints **negative-mining coverage** — a different statistic from the eval slot coverage reported by `eval_concept_ppl_v2.py`.

In [ ]:
%cd {REPO_DIR}

def build_contra(source, out_dir, strategy='all'):
    cmd = (f"python build_contrastive_dataset.py"
           f" --syn_train {SYN_CONCEPT_TRAIN_C} --syn_val {SYN_CONCEPT_VAL_C}"
           f" --hyp_train {HYP_CONCEPT_TRAIN_C} --hyp_val {HYP_CONCEPT_VAL_C}"
           f" --source {source} --strategy {strategy} --max_negatives 10"
           f" --output_dir {out_dir}")
    print(f'\n=== source={source} strategy={strategy} -> {out_dir}')
    !{cmd}

build_contra('both', CONTRA_DIRS['merged'])
build_contra('syn',  CONTRA_DIRS['syn_only'])
build_contra('hyp',  CONTRA_DIRS['hyp_only'])
for s in ABLATION_STRATEGIES:
    build_contra('both', CONTRA_DIRS[f'ablation_{s}'], strategy=s)

In [ ]:
# Sanity: per-dataset negative-mining coverage; wrong_sense MUST be > 0 now.
import ast

def train_csv_of(name):
    d = CONTRA_DIRS[name]
    for f in ['contrastive_train.csv', 'syn_contrastive_train.csv', 'hyp_contrastive_train.csv']:
        p = os.path.join(d, f)
        if os.path.exists(p):
            return p
    return None

rows = []
for name in CONTRA_DIRS:
    p = train_csv_of(name)
    if p is None:
        rows.append({'dataset': name, 'rows': 0, 'mining_coverage': '-', 'avg_negs': '-'})
        continue
    df = pd.read_csv(p)
    negs = df['negatives'].apply(lambda s: ast.literal_eval(str(s)))
    covered = (negs.str.len() > 0)
    rows.append({'dataset': name, 'rows': len(df),
                 'mining_coverage': f'{100 * covered.mean():.1f}%',
                 'avg_negs': f'{negs[covered].str.len().mean():.1f}' if covered.any() else '0'})
cov_df = pd.DataFrame(rows)
display(cov_df)

ws = cov_df.loc[cov_df.dataset == 'ablation_wrong_sense', 'mining_coverage'].iloc[0]
assert ws not in ('0.0%', '-'), 'wrong_sense coverage is still zero — builder fix not applied!'
assert cov_df.loc[cov_df.dataset == 'ablation_none', 'mining_coverage'].iloc[0] == '0.0%', \
    "'none' control should have zero negatives"
print(f'\nOK — wrong_sense mining coverage: {ws} (was 0% in the June round)')

---
## Phase 2 — Retrain every model on the clean splits (Priority 2)

**Why retrain the baselines too?** The clean resplit moves contexts between train and val. A checkpoint trained on the *old* train side may have seen contexts that are now in the *clean* val side — evaluating old checkpoints on the new val set would just reintroduce the leak. Every number in this notebook comes from a model trained on `youtube_clean`.

Runs (~15–45 min each on T4/L4, dataset is ~2K rows): CLM → Syn-NCP → Hyp-NCP → Diff-NCP → 3 contrastive variants. All use `--seed 42`.

In [ ]:
# 2a. Standard CLM baseline (vanilla objective on augmented clean text)
%cd {SCRIPTS_DIR}
cmd = (f"python3 run_clm.py"
       f" --model_name_or_path {MODEL_LOCAL_PATH}"
       f" --train_file {SYN_TXT_TRAIN_C}"
       f" --validation_file {SYN_TXT_VAL_C}"
       f" --save_total_limit 1 --seed 42"
       f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
       f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
       f" --overwrite_output_dir --do_train --do_eval"
       f" --output_dir {CKPTS['clm']}")
!{cmd}

In [ ]:
# 2b. Synonym NCP (original objective, clean syn context CSV)
%cd {SCRIPTS_DIR}
cmd = (f"python3 run_clm_syn_custom_loss.py"
       f" --model_name_or_path {MODEL_LOCAL_PATH}"
       f" --train_file {SYN_CONCEPT_TRAIN_C}"
       f" --validation_file {SYN_CONCEPT_VAL_C}"
       f" --save_total_limit 1 --seed 42"
       f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
       f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
       f" --overwrite_output_dir --do_train --do_eval"
       f" --output_dir {CKPTS['syn_ncp']}")
!{cmd}

In [ ]:
# 2c. Hypernym NCP (original objective, clean hyp dict CSV) — OPTIONAL baseline;
# skip if budget is tight, but the hyp-side story is stronger with it.
%cd {SCRIPTS_DIR}
cmd = (f"python3 run_clm_hyp_custom_loss.py"
       f" --model_name_or_path {MODEL_LOCAL_PATH}"
       f" --train_file {HYP_DICT_TRAIN_C}"
       f" --validation_file {HYP_DICT_VAL_C}"
       f" --save_total_limit 1 --seed 42"
       f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
       f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
       f" --overwrite_output_dir --do_train --do_eval"
       f" --output_dir {CKPTS['hyp_ncp']}")
!{cmd}

In [ ]:
# 2d. Differentiable NCP (α=1.0, clean syn context CSV)
%cd {SCRIPTS_DIR}
cmd = (f"python3 run_clm_differentiable_ncp.py"
       f" --model_name_or_path {MODEL_LOCAL_PATH}"
       f" --train_file {SYN_CONCEPT_TRAIN_C}"
       f" --validation_file {SYN_CONCEPT_VAL_C}"
       f" --ncp_alpha 1.0 --seed 42"
       f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
       f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
       f" --overwrite_output_dir --do_train --do_eval"
       f" --output_dir {CKPTS['diff_ncp']}")
!{cmd}

In [ ]:
# 2e. Contrastive variants: merged, syn-only, hyp-only (α=0.5, β=1.0, seed 42)
%cd {SCRIPTS_DIR}

def contra_files(name):
    d = CONTRA_DIRS[name]
    for tr, va in [('contrastive_train.csv', 'contrastive_val.csv'),
                   ('syn_contrastive_train.csv', 'syn_contrastive_val.csv'),
                   ('hyp_contrastive_train.csv', 'hyp_contrastive_val.csv')]:
        if os.path.exists(os.path.join(d, tr)):
            return os.path.join(d, tr), os.path.join(d, va)
    raise FileNotFoundError(d)

def train_contrastive(ckpt_dir, train_f, val_f, seed=42):
    cmd = (f"python run_clm_contrastive.py"
           f" --model_name_or_path {MODEL_LOCAL_PATH}"
           f" --train_file {train_f} --validation_file {val_f}"
           f" --ncp_alpha 0.5 --contrast_beta 1.0 --seed {seed}"
           f" --gradient_accumulation_steps 8 --per_device_train_batch_size 2 --per_device_eval_batch_size 2"
           f" --torch_dtype bfloat16 --bf16 True --block_size 128 --auto_find_batch_size True"
           f" --save_total_limit 1 --overwrite_output_dir --do_train --do_eval"
           f" --output_dir {ckpt_dir}")
    !{cmd}

for name in ['merged', 'syn_only', 'hyp_only']:
    tr, va = contra_files(name)
    print(f'\n{"="*70}\nTraining contrastive_{name}\n{"="*70}')
    train_contrastive(CKPTS[f'contrastive_{name}'], tr, va)

---
## Phase 3 — Revised dual evaluation (Task 5R)

`eval_concept_ppl_v2.py`: one canonical tokenizer for all checkpoints, **no eval-time embedding resize**, candidates scored as **in-context continuations** (multi-token supported), 95% bootstrap CIs.

**Do not compare these concept PPLs to the June numbers.** The June metric scored bare-word token ids — the same convention the trainers optimize (circular) — on leaked splits. Expect concept PPLs here to be much larger and more honest.

In [ ]:
# 3a. Dual eval on the clean SYN concept val set
EVAL_KEYS = ['clm', 'syn_ncp', 'hyp_ncp', 'diff_ncp',
             'contrastive_merged', 'contrastive_syn_only', 'contrastive_hyp_only']
DUAL_SYN_JSON = f'{RESULTS_DIR}/dual_eval_v2_syn.json'

ck = existing(EVAL_KEYS)
print(f'Evaluating {len(ck)} checkpoints: {list(ck)}')
%cd {SCRIPTS_DIR}
cmd = (f"python eval_concept_ppl_v2.py"
       f" --checkpoints {' '.join(ck.values())}"
       f" --tokenizer_path {TOKENIZER_PATH}"
       f" --concept_csv {SYN_CONCEPT_VAL_C}"
       f" --vanilla_val {VANILLA_VAL_C}"
       f" --results_json {DUAL_SYN_JSON}")
!{cmd}

In [ ]:
# 3b. Dual eval on the clean HYP concept val set
DUAL_HYP_JSON = f'{RESULTS_DIR}/dual_eval_v2_hyp.json'

%cd {SCRIPTS_DIR}
cmd = (f"python eval_concept_ppl_v2.py"
       f" --checkpoints {' '.join(ck.values())}"
       f" --tokenizer_path {TOKENIZER_PATH}"
       f" --concept_csv {HYP_CONCEPT_VAL_C}"
       f" --vanilla_val {VANILLA_VAL_C}"
       f" --results_json {DUAL_HYP_JSON}")
!{cmd}

In [ ]:
# 3c. Results — side-by-side with bootstrap CIs (self-contained cell)
import json, os
import pandas as pd

RESULTS_DIR   = '/content/drive/MyDrive/concept_aware_outputs/clean_results'
DUAL_SYN_JSON = f'{RESULTS_DIR}/dual_eval_v2_syn.json'
DUAL_HYP_JSON = f'{RESULTS_DIR}/dual_eval_v2_hyp.json'

def dual_table(path, tag):
    if not os.path.exists(path):
        return pd.DataFrame()
    rows = []
    for r in json.load(open(path)):
        if 'error' in r or 'concept_error' in r:
            continue
        ci = r['concept_ppl_ci95']
        rows.append({
            'Model': os.path.basename(r['checkpoint'].rstrip('/')),
            'NTP PPL': r['ntp_ppl'],
            'NTP Acc': f"{r['ntp_acc']:.3f}",
            f'{tag} Concept PPL': r['concept_ppl'],
            f'{tag} 95% CI': f"[{ci[0]}, {ci[1]}]",
            f'{tag} Set Mass': r['concept_set_mass_mean'],
            'Slots': r['concept_n_rows_scored'],
            'Slot Cov': f"{r['eval_slot_coverage_pct']:.1f}%",
        })
    return pd.DataFrame(rows)

df_syn = dual_table(DUAL_SYN_JSON, 'Syn')
df_hyp = dual_table(DUAL_HYP_JSON, 'Hyp')
print('=== Task 5R: Synonym concept eval (clean splits, v2 metric) ===')
display(df_syn)
print('\n=== Task 5R: Hypernym concept eval (clean splits, v2 metric) ===')
display(df_hyp)

---
## Phase 4 — Fair single-source comparison (Task 6R)

The June round only controlled the syn side (and its "merged wins" conclusion was the leak). With clean splits and a **hyp-only** model, both directions are controlled:

- **Objective test**: contrastive syn-only vs Syn-NCP (same data, different objective).
- **Data test**: merged vs syn-only vs hyp-only (same objective, different data). Any merged advantage now is real, not leakage.

In [ ]:
# Task 6R comparison table (self-contained)
import json, os
import pandas as pd

RESULTS_DIR = '/content/drive/MyDrive/concept_aware_outputs/clean_results'
FOCUS = ['standard_clm', 'syn_ncp', 'contrastive_syn_only', 'contrastive_hyp_only', 'contrastive_merged']

def by_ckpt(path):
    if not os.path.exists(path):
        return {}
    return {os.path.basename(r['checkpoint'].rstrip('/')): r
            for r in json.load(open(path)) if 'error' not in r and 'concept_error' not in r}

syn = by_ckpt(f'{RESULTS_DIR}/dual_eval_v2_syn.json')
hyp = by_ckpt(f'{RESULTS_DIR}/dual_eval_v2_hyp.json')

rows = []
for k in FOCUS:
    s, h = syn.get(k, {}), hyp.get(k, {})
    if not s and not h:
        continue
    rows.append({
        'Model': k,
        'NTP PPL': s.get('ntp_ppl', '-'),
        'Syn Concept PPL': s.get('concept_ppl', '-'),
        'Syn CI': str(s.get('concept_ppl_ci95', '-')),
        'Hyp Concept PPL': h.get('concept_ppl', '-'),
        'Hyp CI': str(h.get('concept_ppl_ci95', '-')),
    })
df6 = pd.DataFrame(rows)
print('=== Task 6R: single-source controls (clean splits) ===')
display(df6)
print('Read: if syn-only contrastive < syn_ncp on Syn Concept PPL (non-overlapping CIs), '
      'the objective helps; if merged < syn-only on BOTH columns, merging genuinely helps.')

---
## Phase 5 — Hard-negative strategy ablation, rerun (Task 8R)

Four models on the clean merged data: `co_hyponym`, `wrong_sense` (**now actually mining wrong senses** — June's run had zero negatives), `same_pos`, and `none` (explicit no-negatives control — what the June "wrong-sense" run accidentally was, kept because it isolates the InfoNCE term's contribution).

In [ ]:
# 5a. Train the four ablation models (~4 GPU-runs)
%cd {SCRIPTS_DIR}
for s in ABLATION_STRATEGIES:
    d = CONTRA_DIRS[f'ablation_{s}']
    tr, va = os.path.join(d, 'contrastive_train.csv'), os.path.join(d, 'contrastive_val.csv')
    if not os.path.exists(tr):
        print(f'SKIP {s}: dataset missing ({tr})')
        continue
    print(f'\n{"="*70}\nTraining ablation: {s}\n{"="*70}')
    train_contrastive(CKPTS[f'ablation_{s}'], tr, va)

In [ ]:
# 5b. Dual eval of ablations + anchors on both clean val sets
ABL_KEYS = ['clm', 'syn_ncp', 'contrastive_merged'] + [f'ablation_{s}' for s in ABLATION_STRATEGIES]
ck8 = existing(ABL_KEYS)

for tag, csv_path in [('syn', SYN_CONCEPT_VAL_C), ('hyp', HYP_CONCEPT_VAL_C)]:
    out = f'{RESULTS_DIR}/task8R_ablation_{tag}.json'
    %cd {SCRIPTS_DIR}
    cmd = (f"python eval_concept_ppl_v2.py"
           f" --checkpoints {' '.join(ck8.values())}"
           f" --tokenizer_path {TOKENIZER_PATH}"
           f" --concept_csv {csv_path}"
           f" --vanilla_val {VANILLA_VAL_C}"
           f" --results_json {out}")
    !{cmd}

In [ ]:
# 5c. Ablation table (self-contained)
import json, os
import pandas as pd

RESULTS_DIR = '/content/drive/MyDrive/concept_aware_outputs/clean_results'
NAMES = {
    'standard_clm':                    'CLM (baseline)',
    'syn_ncp':                         'Syn-NCP (original)',
    'contrastive_merged':              'Contrastive — ALL strategies',
    'contrastive_ablation_co_hyponym': 'Contrastive — co-hyponym only',
    'contrastive_ablation_wrong_sense':'Contrastive — wrong-sense only (FIXED)',
    'contrastive_ablation_same_pos':   'Contrastive — same-POS only',
    'contrastive_ablation_none':       'Contrastive — NO negatives (control)',
}

def by_ckpt(path):
    if not os.path.exists(path):
        return {}
    return {os.path.basename(r['checkpoint'].rstrip('/')): r
            for r in json.load(open(path)) if 'error' not in r and 'concept_error' not in r}

syn = by_ckpt(f'{RESULTS_DIR}/task8R_ablation_syn.json')
hyp = by_ckpt(f'{RESULTS_DIR}/task8R_ablation_hyp.json')

rows = []
for key, label in NAMES.items():
    s, h = syn.get(key, {}), hyp.get(key, {})
    if not s and not h:
        continue
    rows.append({'Model': label,
                 'NTP PPL': s.get('ntp_ppl', '-'), 'NTP Acc': s.get('ntp_acc', '-'),
                 'Syn Concept PPL': s.get('concept_ppl', '-'), 'Syn CI': str(s.get('concept_ppl_ci95', '-')),
                 'Hyp Concept PPL': h.get('concept_ppl', '-')})
print('=== Task 8R: hard-negative strategy ablation (clean splits, fixed mining) ===')
display(pd.DataFrame(rows))
print("Key contrast vs June: 'wrong-sense only' now actually trains against wrong-sense negatives; "
      "'NO negatives' shows what the June wrong-sense run really measured.")

---
## Phase 6 — Insurance seed (Priority 3, ~1 GPU-hour)

One extra seed of the **headline model only** (merged contrastive). If seed 123 lands in the same regime as seed 42, the narrative is de-risked before paper writing; the full 3-seed × 3-model grid stays deferred to camera-ready.

In [ ]:
# 6a. Retrain merged contrastive with seed 123 and evaluate
tr, va = contra_files('merged')
%cd {SCRIPTS_DIR}
train_contrastive(CKPTS['contrastive_merged_seed123'], tr, va, seed=123)

SEED_JSON = f'{RESULTS_DIR}/seed_check.json'
ck_seed = existing(['contrastive_merged', 'contrastive_merged_seed123'])
cmd = (f"python eval_concept_ppl_v2.py"
       f" --checkpoints {' '.join(ck_seed.values())}"
       f" --tokenizer_path {TOKENIZER_PATH}"
       f" --concept_csv {SYN_CONCEPT_VAL_C}"
       f" --vanilla_val {VANILLA_VAL_C}"
       f" --results_json {SEED_JSON}")
!{cmd}

In [ ]:
# 6b. Seed comparison (self-contained)
import json, os
import pandas as pd

SEED_JSON = '/content/drive/MyDrive/concept_aware_outputs/clean_results/seed_check.json'
if os.path.exists(SEED_JSON):
    rows = []
    for r in json.load(open(SEED_JSON)):
        if 'error' in r:
            continue
        rows.append({'Model': os.path.basename(r['checkpoint'].rstrip('/')),
                     'NTP PPL': r['ntp_ppl'], 'NTP Acc': r['ntp_acc'],
                     'Syn Concept PPL': r['concept_ppl'],
                     '95% CI': str(r['concept_ppl_ci95'])})
    display(pd.DataFrame(rows))
    print('If the two rows have overlapping CIs / same ordering vs baselines, the result is seed-stable.')
else:
    print('Run 6a first.')

---
## Phase 7 — Downstream evaluation, redesigned (Task 7R, Priority 4)

June's finding: **full fine-tuning saturates** (SNLI ≈ 0.89, SPAM ≈ 0.99 for every model) — it cannot discriminate between pre-training objectives, exactly as the paper's own Limitations section warns. Redesign:

1. **Linear probe = primary metric** (representation quality in isolation).
2. **Low-resource fine-tuning** (n ∈ {100, 500, 1000}): if concept training helps downstream, it should show where the task head cannot simply relearn everything.
3. SPAM kept as an optional sanity task only (saturated; near-ceiling for all models).

In [ ]:
# 7a. SNLI linear probe — primary transfer metric
DS_KEYS = ['clm', 'syn_ncp', 'hyp_ncp', 'diff_ncp',
           'contrastive_merged', 'contrastive_syn_only', 'contrastive_hyp_only']
ck7 = existing(DS_KEYS)
SNLI_PROBE_JSON = f'{RESULTS_DIR}/snli_probe_v2.json'
DOWNSTREAM_DIR  = f'{OUTPUT_ROOT}/downstream_clean'

%cd {SCRIPTS_DIR}
cmd = (f"python run_downstream_eval.py"
       f" --checkpoints {' '.join(ck7.values())}"
       f" --task snli --freeze_base"
       f" --max_train_samples 20000 --num_epochs 5"
       f" --output_dir {DOWNSTREAM_DIR}"
       f" --results_json {SNLI_PROBE_JSON}")
!{cmd}

In [ ]:
# 7b. SNLI low-resource fine-tuning curve (n = 100 / 500 / 1000)
for n in [100, 500, 1000]:
    out = f'{RESULTS_DIR}/snli_lowres_{n}.json'
    print(f'\n{"="*70}\nSNLI full fine-tune with n={n} training examples\n{"="*70}')
    %cd {SCRIPTS_DIR}
    cmd = (f"python run_downstream_eval.py"
           f" --checkpoints {' '.join(ck7.values())}"
           f" --task snli"
           f" --max_train_samples {n} --num_epochs 10"
           f" --output_dir {DOWNSTREAM_DIR}"
           f" --results_json {out}")
    !{cmd}

In [ ]:
# 7c. OPTIONAL: SPAM linear probe (saturated task — sanity check only)
SPAM_PROBE_JSON = f'{RESULTS_DIR}/spam_probe_v2.json'
%cd {SCRIPTS_DIR}
cmd = (f"python run_downstream_eval.py"
       f" --checkpoints {' '.join(ck7.values())}"
       f" --task spam --freeze_base --num_epochs 5"
       f" --output_dir {DOWNSTREAM_DIR}"
       f" --results_json {SPAM_PROBE_JSON}")
!{cmd}

In [ ]:
# 7d. Downstream results (self-contained)
import json, os
import pandas as pd

RESULTS_DIR = '/content/drive/MyDrive/concept_aware_outputs/clean_results'

def load_ds(path, mode):
    if not os.path.exists(path):
        return []
    return [{'Model': os.path.basename(r['checkpoint'].rstrip('/')), 'Mode': mode,
             'Accuracy': r.get('accuracy'), 'F1': r.get('f1')}
            for r in json.load(open(path)) if 'error' not in r]

frames = load_ds(f'{RESULTS_DIR}/snli_probe_v2.json', 'SNLI probe')
for n in [100, 500, 1000]:
    frames += load_ds(f'{RESULTS_DIR}/snli_lowres_{n}.json', f'SNLI FT n={n}')
frames += load_ds(f'{RESULTS_DIR}/spam_probe_v2.json', 'SPAM probe')

df7 = pd.DataFrame(frames)
if not df7.empty:
    print('=== Task 7R: downstream (probe-first + low-resource) ===')
    display(df7.pivot_table(index='Model', columns='Mode', values='Accuracy', aggfunc='first').round(4))
else:
    print('No downstream results yet — run 7a/7b first.')

---
## Phase 8 — Master table (all revised results)

In [ ]:
# Master comparison across all phases (self-contained)
import json, os
import pandas as pd

RESULTS_DIR = '/content/drive/MyDrive/concept_aware_outputs/clean_results'
OUTPUT_ROOT = '/content/drive/MyDrive/concept_aware_outputs'

def by_ckpt(path):
    if not os.path.exists(path):
        return {}
    return {os.path.basename(r['checkpoint'].rstrip('/')): r
            for r in json.load(open(path)) if 'error' not in r and 'concept_error' not in r}

syn  = by_ckpt(f'{RESULTS_DIR}/dual_eval_v2_syn.json')
hyp  = by_ckpt(f'{RESULTS_DIR}/dual_eval_v2_hyp.json')
abl_syn = by_ckpt(f'{RESULTS_DIR}/task8R_ablation_syn.json')
probe = by_ckpt(f'{RESULTS_DIR}/snli_probe_v2.json')
low500 = by_ckpt(f'{RESULTS_DIR}/snli_lowres_500.json')

ORDER = [
    ('standard_clm',                     'Standard CLM'),
    ('syn_ncp',                          'Synonym NCP'),
    ('hyp_ncp',                          'Hypernym NCP'),
    ('diff_ncp',                         'Differentiable NCP'),
    ('contrastive_merged',               'Contrastive (merged)'),
    ('contrastive_syn_only',             'Contrastive (syn-only)'),
    ('contrastive_hyp_only',             'Contrastive (hyp-only)'),
    ('contrastive_ablation_co_hyponym',  '  ablation: co-hyponym'),
    ('contrastive_ablation_wrong_sense', '  ablation: wrong-sense (fixed)'),
    ('contrastive_ablation_same_pos',    '  ablation: same-POS'),
    ('contrastive_ablation_none',        '  ablation: no negatives'),
]

rows = []
for key, label in ORDER:
    s = syn.get(key) or abl_syn.get(key) or {}
    h = hyp.get(key, {})
    p = probe.get(key, {})
    l = low500.get(key, {})
    if not (s or h or p):
        continue
    rows.append({
        'Model': label,
        'NTP PPL': s.get('ntp_ppl', '-'),
        'NTP Acc': s.get('ntp_acc', '-'),
        'Syn cPPL': s.get('concept_ppl', '-'),
        'Syn cPPL CI': str(s.get('concept_ppl_ci95', '-')),
        'Hyp cPPL': h.get('concept_ppl', '-'),
        'SNLI probe': p.get('accuracy', '-'),
        'SNLI FT@500': l.get('accuracy', '-'),
    })

df_master = pd.DataFrame(rows)
print('=== MASTER TABLE — revised round (clean splits, v2 metric) ===')
display(df_master)

master_csv = f'{OUTPUT_ROOT}/master_comparison_revised.csv'
df_master.to_csv(master_csv, index=False)
print(f'Saved: {master_csv}')

---
## Deferred (by design — budget priorities 5–6)

- **Full 3-seed × 3-model grid** (CLM, Syn-NCP, Contrastive-merged): run at paper-writing time on the final configuration; report mean ± std. The Phase 6 insurance seed de-risks this.
- **Pythia-1.4B**: only after the Llama-1B story is solid (Chen's ordering). Remember Pythia's BOS/EOS ids differ from Llama's.
- **Manual audit of hyp-val positives** (~50 slots, 30 min, no GPU): are the hypernym "positives" actually valid in context? Do this before submitting — a reviewer will.
- **Cross-domain robustness** (train YouTube → eval news/arXiv with v2): engages the paper's central Table-2 claim; cheap eval-only follow-up once checkpoints exist.

### Provenance
- Leak discovered + fixed 2026-07-06; splits rebuilt with `rebuild_clean_splits.py --seed 42 --val_frac 0.12`.
- Wrong-sense mining fixed the same day (synonym-intersection WSD + hierarchy filter).
- June numbers (`research_tasks_5_8.ipynb`) are retained for the record but are **superseded by this notebook**.